<a href="https://colab.research.google.com/github/andreamarin/senate-publications-analysis/blob/add%2Fbertopic-analysis/bertopic_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set up

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!git clone https://github.com/andreamarin/senate-publications-analysis.git

Cloning into 'senate-publications-analysis'...
remote: Enumerating objects: 337, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 337 (delta 73), reused 43 (delta 23), pack-reused 204 (from 1)
Receiving objects: 100% (337/337), 2.11 MiB | 10.49 MiB/s, done.
Resolving deltas: 100% (181/181), done.


In [14]:
%cd senate-publications-analysis/nlp_classification/

/content/senate-publications-analysis/nlp_classification


In [3]:
!git checkout add/bertopic-analysis

Branch 'add/bertopic-analysis' set up to track remote branch 'add/bertopic-analysis' from 'origin'.
Switched to a new branch 'add/bertopic-analysis'


In [5]:
%mkdir config

In [6]:
%cp ../../drive/MyDrive/tesis/code/config/* ./config/.

In [7]:
%ls config

bot-cert.pem


In [16]:
!git pull

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.67 KiB | 856.00 KiB/s, done.
From https://github.com/andreamarin/senate-publications-analysis
   df7c6ed..e0fcb27  add/bertopic-analysis -> origin/add/bertopic-analysis
Updating df7c6ed..e0fcb27
Fast-forward
 nlp_classification/utils/bertopic_model_builder.py | 76 ++++++++++++++++++++++
 1 file changed, 76 insertions(+)


In [8]:
!pip install -r bert_requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331

In [1]:
! python -m spacy download es_core_news_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.0/568.0 MB 2.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
!curl ipecho.net/plain

34.75.224.26

In [3]:
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 54.8 gigabytes of available RAM

You are using a high-RAM runtime!


# Imports

In [4]:
import sys
sys.path.append('/content/senate-publications-analysis/nlp_classification')

In [17]:
import importlib
import pandas as pd

In [5]:
import utils.db as db
import utils.bertopic_evaluator as evaluator
import utils.bertopic_model_builder as model_builder

In [18]:

importlib.reload(model_builder)

<module 'utils.bertopic_model_builder' from '/content/senate-publications-analysis/nlp_classification/utils/bertopic_model_builder.py'>

# Load data

In [7]:
conn = db.connect_mongo_db("senate-publication")

In [11]:
publications_cursor = conn.unique_publications.find(
    projection=["clean_summary", "type"]
  )
publications_df = pd.DataFrame(publications_cursor)
publications_df.head()

,_id,type,clean_summary
0,0018ac0cd878129887392b0f49455372,proposicion,"- La Congreso_Union exhorta, respetuosamente, ..."
1,00235c72fe5869865b614ea03124fde2,iniciativa,Propone establecer que los riesgos o daños a l...
2,0060157909fe1ded5485fcab55173969,iniciativa,Propone establecer que el Magistrado instructo...
3,0082043cbeb4f1f5c52f1deeb0f24316,iniciativa,Propone establecer como objeto de este ordenam...
4,00a479d44f0acb53cc4be46248d056bf,proposicion,"ÙNICO. La Congreso_Union exhorta, respetuosame..."


# Run models

In [12]:
base_params = {
    "texts_df": publications_df,
    "text_column": "clean_summary",
    "folder_name": "senate",
    "base_path": "/drive/MyDrive/tesis/bertopic_models"
}

In [23]:
params = [
    {
        "embedding_config": model_builder.EmbeddingConfig(
            embedding_model="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
            max_words=80,
            spacy_model="es_core_news_lg",
            document_representation=model_builder.DocumentRepresentation.FULL_TEXT,
        ),
        "umap_config": model_builder.UMAPConfig(
            n_neighbors=15,
            metric="cosine",
            min_dist=0.1
        ),
        "hdbscan_config": model_builder.HDBSCANConfig(
            min_cluster_size=15,
        ),
        "verbose": True
    }
]

In [24]:
topic_model = model_builder.BerTopicModelBuilder(
    **base_params,
    **params[0]
)

In [ ]:
topic_model.fit_transform()

2026-04-25 23:32:57,461 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Starting fit_transform.
2026-04-25 23:32:57,463 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Embeddings cache not found. Computing embeddings.
2026-04-25 23:32:57,464 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loading embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-04-25 23:33:05,605 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Document representation: full_text
2026-04-25 23:33:05,605 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Encoding 9351 full documents.


Batches:   0%|          | 0/293 [00:00<?, ?it/s]